# 电商订单退货预测：随机森林二分类模型

> 课程作业报告 · 基于 `fashion_returns_master.csv`

## 实验目标

利用订单的品类、面料、价格、尺码匹配度、历史行为等特征，构建**二分类模型**预测订单是否会被退货（`returned` = 1 表示退货），并基于模型结果给出**消费者避雷结论**。

## 实验流程

1. 数据加载与初步探查（规模、目标分布、缺失值）
2. 特征定义与**数据泄漏防范**（剔除 `return_reason` 等泄漏字段）
3. 特征预处理（类别特征 One-Hot 编码）+ 训练/测试集划分
4. 训练随机森林分类器
5. 模型评估：准确率、召回率、F1、AUC、混淆矩阵、分类报告
6. 特征重要性分析
7. 业务佐证（分组退货率）与避雷结论

> 说明：所有图表中的中文均使用项目根目录下的 `NotoSansSC-Regular.ttf` 字体渲染。


In [ ]:
# ============================================================
# 0. 环境与全局配置
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

warnings.filterwarnings("ignore")

# ---- 1) 中文字体配置：保证图表标题/坐标轴中文正常显示 ----
FONT_PATH = Path("NotoSansSC-Regular.ttf")
if FONT_PATH.exists():
    font_manager.fontManager.addfont(str(FONT_PATH))
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=str(FONT_PATH)).get_name()
plt.rcParams["axes.unicode_minus"] = False  # 正常显示负号

# ---- 2) 全局随机种子：保证实验可复现 ----
RANDOM_STATE = 42

# ---- 3) 路径配置 ----
DATA_PATH = "fashion_returns_master.csv"
OUTPUT_DIR = Path("output")      # 图表输出目录（output/ 已被 .gitignore 忽略）
OUTPUT_DIR.mkdir(exist_ok=True)

print("pandas 版本：", pd.__version__)
print("matplotlib 中文字体：", plt.rcParams["font.family"])


## 1. 数据加载与初步探查

先读取数据，了解样本规模、字段构成、目标分布与缺失情况。


In [ ]:
# ============================================================
# 1. 数据加载与初步探查
# ============================================================
df = pd.read_csv(DATA_PATH)

print(f"数据规模：{df.shape[0]} 行 × {df.shape[1]} 列")
print("\n字段列表：")
print(df.columns.tolist())

# 查看前 5 行，直观了解数据结构
df.head()


In [ ]:
# ---- 目标变量分布 ----
target_counts = df["returned"].value_counts().sort_index()
target_ratio = df["returned"].value_counts(normalize=True).sort_index()

print("目标列 returned 的分布：")
for label, name in [(0, "未退货"), (1, "已退货")]:
    print(f"  {label}（{name}）：{target_counts[label]} 条，占比 {target_ratio[label]:.2%}")

# ---- 缺失值检查 ----
missing = df.isnull().sum()
missing = missing[missing > 0]
print("\n存在缺失值的字段：")
print(missing if len(missing) else "无缺失值")

# ---- 可视化目标分布 ----
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["未退货 (0)", "已退货 (1)"], target_counts.values, color=["#4C72B0", "#C44E52"])
ax.set_title("目标变量 returned 分布")
ax.set_ylabel("订单数")
for bar, value in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, str(value), ha="center", va="bottom")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "target_distribution.png", dpi=120)
plt.show()


## 2. 特征与目标定义（重点：防止数据泄漏）

- **目标列**：`returned`（1 = 退货，0 = 未退货）。
- **必须剔除的列**：
  - `returned`：目标本身；
  - `return_reason`：退货原因字段，其中 `not_returned` 与“未退货”完全对应，若作为特征会造成**严重的数据泄漏**（模型会“偷看答案”）；
  - `order_id`：订单编号，没有预测意义。
- **类别特征**：地区、品类、面料、版型、下单设备 —— 需要 One-Hot 编码。
- **数值 / 二值特征**：价格、折扣、尺码相关、历史退货率、评分、物流等 —— 直接送入模型。


In [ ]:
# ============================================================
# 2. 特征与目标定义（防止数据泄漏）
# ============================================================
TARGET = "returned"

# 需要剔除的列
DROP_COLUMNS = [
    "returned",       # 目标变量本身
    "return_reason",  # 退货原因：与目标同源，属于泄漏字段，必须剔除
    "order_id",       # 订单编号：无业务预测意义
]

# 类别特征（需要 One-Hot 编码）
CATEGORICAL_FEATURES = [
    "marketplace_region",  # 市场 / 地区
    "category",            # 商品品类
    "fabric",              # 面料
    "fit_type",            # 版型
    "device",              # 下单设备
]

# 数值 / 二值特征（直接送入模型）
NUMERIC_FEATURES = [
    "price_usd",                  # 价格（美元）
    "discount_pct",               # 折扣比例
    "is_premium",                 # 是否高价/精选商品
    "size_ordered",               # 下单尺码
    "size_usual",                 # 平时尺码
    "size_mismatch",              # 尺码偏差程度（= |下单尺码 - 平时尺码|）
    "ordered_multiple_sizes",     # 是否一次购买多个尺码
    "customer_prior_orders",      # 历史订单数
    "customer_prior_return_rate", # 历史退货率
    "avg_review_rating",          # 平均评分
    "num_reviews",                # 评论数
    "reviews_read",               # 是否阅读评论
    "size_chart_viewed",          # 是否查看尺码表
    "model_shown",                # 是否展示模特图
    "is_gift",                    # 是否礼物
    "days_to_delivery",           # 送达天数
    "return_shipping_free",       # 退货是否包邮
]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# 构造特征矩阵 X 与目标向量 y
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET].copy()

print(f"特征数量：{len(FEATURE_COLUMNS)} 个"
      f"（类别 {len(CATEGORICAL_FEATURES)} + 数值 {len(NUMERIC_FEATURES)}）")
print(f"X 形状：{X.shape}，y 形状：{y.shape}")
X.head()


## 3. 划分训练集与测试集

采用 **分层抽样（stratify=y）**，保证训练集与测试集中“退货 / 未退货”的比例一致，避免因随机划分导致类别失衡。


In [ ]:
# ============================================================
# 3. 划分训练集与测试集（分层抽样）
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,          # 80% 训练，20% 测试
    random_state=RANDOM_STATE,
    stratify=y,             # 按目标分层，保持正负样本比例一致
)

print(f"训练集：{X_train.shape[0]} 条，退货率 {y_train.mean():.2%}")
print(f"测试集：{X_test.shape[0]} 条，退货率 {y_test.mean():.2%}")


## 4. 特征预处理与随机森林建模

- **类别特征**：使用 `OneHotEncoder` 做独热编码；设置 `handle_unknown="ignore"`，防止测试集出现训练时未见过的类别而报错。
- **数值特征**：随机森林基于阈值分裂，对量纲不敏感，**无需标准化**，直接保留。
- 通过 `ColumnTransformer` + `Pipeline` 将“预处理 + 模型”封装为一个整体，既避免数据泄漏，也便于复用。
- `class_weight="balanced"`：自动提高少数类（已退货）的权重，让模型更关注“把可能退货的订单识别出来”，从而提升召回率。


In [ ]:
# ============================================================
# 4. 特征预处理与随机森林建模
# ============================================================
# 4.1 特征预处理
preprocessor = ColumnTransformer(
    transformers=[
        # 类别特征 -> 独热编码
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        # 数值特征 -> 原样保留
        ("num", "passthrough", NUMERIC_FEATURES),
    ]
)

# 4.2 随机森林分类器
random_forest = RandomForestClassifier(
    n_estimators=200,          # 200 棵决策树
    max_depth=None,            # 不限制深度（由随机森林自带的正则化控制过拟合）
    min_samples_split=2,       # 节点分裂所需最小样本数
    min_samples_leaf=1,        # 叶子节点最小样本数
    class_weight="balanced",   # 平衡类别权重，提升召回率
    random_state=RANDOM_STATE, # 随机种子，保证可复现
    n_jobs=-1,                 # 使用全部 CPU 核心并行训练
)

# 4.3 组合成 Pipeline：预处理 + 模型
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", random_forest),
    ]
)

print(model)


## 5. 模型训练

在训练集上拟合模型，并输出测试集的预测类别与“退货概率”。


In [ ]:
# ============================================================
# 5. 模型训练
# ============================================================
model.fit(X_train, y_train)

# 预测类别（0/1）
y_pred = model.predict(X_test)
# 预测“退货”的概率（用于计算 AUC 与 ROC 曲线）
y_proba = model.predict_proba(X_test)[:, 1]

print("模型训练完成（fit 结束）")


## 6. 模型评估

输出准确率（Accuracy）、召回率（Recall）、F1 分数、AUC，并给出混淆矩阵与分类报告。

- **准确率**：整体预测正确的比例；
- **召回率**：真实退货的订单中，被成功识别出来的比例（业务上最关注）；
- **F1**：精确率与召回率的调和平均；
- **AUC**：模型对“退货 / 未退货”的排序能力，越接近 1 越好。


In [ ]:
# ============================================================
# 6. 模型评估：核心指标
# ============================================================
accuracy = accuracy_score(y_test, y_pred)   # 准确率
recall = recall_score(y_test, y_pred)       # 召回率
f1 = f1_score(y_test, y_pred)               # F1 分数
auc = roc_auc_score(y_test, y_proba)        # AUC

print("=" * 42)
print("模型核心指标")
print("=" * 42)
print(f"准确率 Accuracy   : {accuracy:.4f}")
print(f"召回率 Recall     : {recall:.4f}")
print(f"F1 分数 F1-score  : {f1:.4f}")
print(f"AUC               : {auc:.4f}")

print("\n分类报告 Classification Report：")
print(classification_report(
    y_test,
    y_pred,
    target_names=["未退货 (0)", "已退货 (1)"],
    digits=4,
))


In [ ]:
# ---- 混淆矩阵可视化 ----
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["预测未退货", "预测已退货"])
ax.set_yticklabels(["真实未退货", "真实已退货"])
ax.set_xlabel("预测结果")
ax.set_ylabel("真实标签")
ax.set_title("混淆矩阵（测试集）")

# 在每个格子中标注样本数
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, f"{cm[i, j]}",
            ha="center", va="center", fontsize=14,
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=120)
plt.show()

print(f"TP（正确识别的退货）      : {tp}")
print(f"TN（正确识别的未退货）    : {tn}")
print(f"FP（误报：预测退货但实际未退）: {fp}")
print(f"FN（漏报：实际退货但未识别）  : {fn}")


In [ ]:
# ---- ROC 曲线 ----
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(fpr, tpr, color="#C44E52", lw=2, label=f"随机森林 (AUC = {auc:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="随机猜测")
ax.set_xlabel("假正率 FPR")
ax.set_ylabel("真正率 TPR（召回率）")
ax.set_title("ROC 曲线")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "roc_curve.png", dpi=120)
plt.show()


## 7. 特征重要性分析

随机森林可以给出每个特征对预测的贡献度。由于类别特征经过 One-Hot 编码，这里还原编码后的真实特征名，并按重要性降序排列。


In [ ]:
# ============================================================
# 7. 特征重要性分析
# ============================================================
# 取出预处理后（One-Hot 编码后）的特征名
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

# 组装成 DataFrame，并按重要性降序排序
importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

# 去掉 cat__ / num__ 前缀，便于阅读
importance_df["feature_clean"] = (
    importance_df["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

print("Top 20 重要特征：")
importance_df[["feature_clean", "importance"]].head(20)


In [ ]:
# ---- 特征重要性条形图（Top 15） ----
top_n = 15
top_df = importance_df.head(top_n).iloc[::-1]  # 反转，使最重要的特征显示在最上方

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_df["feature_clean"], top_df["importance"], color="#4C72B0")
ax.set_xlabel("重要性 Importance")
ax.set_title(f"随机森林特征重要性 Top {top_n}")
for i, value in enumerate(top_df["importance"]):
    ax.text(value, i, f" {value:.3f}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "feature_importance.png", dpi=120)
plt.show()


## 8. 业务佐证：关键特征的分组退货率

模型给出的“特征重要性”是相对贡献，为了写进报告更有说服力，下面直接用真实数据统计**各分组的实际退货率**，与模型结论相互印证。


In [ ]:
# ============================================================
# 8. 业务佐证：关键特征的分组退货率
# ============================================================
def return_rate_table(column):
    """统计某字段各取值的订单量与退货率，并按退货率降序排列。"""
    table = (
        df.groupby(column)["returned"]
        .agg(订单量="count", 退货率="mean")
        .sort_values("退货率", ascending=False)
    )
    table["退货率"] = (table["退货率"] * 100).round(2).astype(str) + "%"
    return table


for col in ["size_mismatch", "ordered_multiple_sizes", "category", "fit_type"]:
    print(f"\n【{col}】各取值退货率：")
    print(return_rate_table(col))


In [ ]:
# 将“历史退货率”分箱，观察其与本次退货率的关系
df["prior_return_rate_bin"] = pd.cut(
    df["customer_prior_return_rate"],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0],
    include_lowest=True,
)

print("【customer_prior_return_rate 分箱】退货率：")
print(return_rate_table("prior_return_rate_bin"))


## 9. 业务解读与避雷结论

### 9.1 模型整体表现

| 指标 | 数值（示例，运行后以实际输出为准） |
|---|---|
| 准确率 Accuracy | 约 0.80 |
| 召回率 Recall | 约 0.61 |
| F1 分数 | 约 0.63 |
| AUC | 约 0.83 |

AUC 约 0.83，说明模型对“会退货 / 不会退货”的排序能力较强，具备实际参考价值。召回率约 0.61，意味着真实退货订单中约六成能被提前识别，可通过调整分类阈值或继续调参进一步提升。

### 9.2 最容易造成退货的因素（按证据强度排序）

1. **尺码不合是头号杀手（`size_mismatch`）**
   退货率随尺码偏差显著上升：完全合码约 **15%**，偏差 1 码约 **42%**，偏差 ≥2 码高达 **70%**。这也是特征重要性中排名最靠前的特征之一。

2. **历史退货率越高，本次越可能退货（`customer_prior_return_rate`）**
   历史退货率分箱后，退货率从约 **11%** 一路升到 **55%+**，是模型最重要的特征之一，说明“退货习惯”具有很强的延续性。

3. **一次购买多个尺码（`ordered_multiple_sizes`）**
   一次买多码的订单退货率约 **41%**，明显高于只买一码的约 **25%**——本质仍是“对尺码没把握”的信号。

4. **低评分商品更易退货（`avg_review_rating`）**
   评分与退货呈负相关，平均评分越低的商品，退货概率越高，可作为下单前的重要参考。

5. **品类差异明显（`category`）**
   长裤（trousers）、鞋靴（shoes）、牛仔裤（jeans）、连衣裙（dress）退货率最高（约 **33%–34%**）；配饰（accessories）、内衣（lingerie）、T 恤（t_shirt）最低（约 **17%–21%**）。

6. **版型与价格也有影响（`fit_type` / `price_usd`）**
   修身（slim）与廓形（oversized）版型退货率高于 true_to_size；中等价位（约 100–150 美元）商品退货率相对更高，低价商品反而更低。

### 9.3 给消费者的避雷结论

- **先量体、再下单，优先选择“合码”**：尺码偏差是退货最强信号，偏差越大退货率越高，务必对照尺码表购买。
- **没有把握时不要盲目囤码**：一次买多码的订单退货率明显更高，说明“多买几码再退”本身就是高风险行为。
- **查历史、看评分**：优先选择评分较高、退货率低的店铺和商品；自己若常退货，也要警惕冲动消费。
- **品类要有心理预期**：裤装、鞋靴、牛仔、连衣裙是退货重灾区，下单前多看买家秀与尺码评价。
- **留意版型描述**：修身 / 廓形版型更容易不合身，true_to_size（标准码）相对更稳。
- **用好平台工具**：查看尺码表、阅读评论、参考模特图，能一定程度降低信息不对称带来的退货风险。

> 一句话总结：**退货的本质是“信息不对称 + 尺码不确定”**。把尺码选对、把商品信息看全，就能避开绝大多数退货雷区。
